## TrustLens — Conversation AI Detection (Baseline)

This notebook adapts the baseline (text classification) to our **TrustLens** project, focusing on **conversation-level AI detection at the turn level**.

### Goal (Sprint 3)
Train a lightweight baseline classifier (Logistic Regression + TF-IDF) **offline** to predict whether each conversation turn is:
- **AI-generated** or
- **Human-written**

### Why this baseline?
- Fast to train and easy to debug
- Strong starting point for classical text classification
- Produces consistent predictions that can later be integrated into the web platform

### Key Output
After training, the notebook generates:
1) A trained model (saved locally in Drive)  
2) Predictions for the evaluation split (CSV)  
3) A basic classification report (Precision / Recall / F1)

### Important Note on Data Splitting

We split the dataset by `dialogue_id` so that:
- All turns of the same dialogue go **either** to train **or** to test  
This avoids leakage and produces a more realistic evaluation.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
TrustLens baseline adapted from the lab template:
- Offline training
- Turn-level conversation classification
"""

from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import LinearSVC

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import text
from sklearn.datasets import load_svmlight_file

import numpy as np
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

from os import path
import os
import pandas as pd
import logging

# Extra utilities (kept minimal and standard)
from sklearn.model_selection import train_test_split
import joblib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:


data_path = "/content/drive/MyDrive/GP2/NewTrustLens/Dataset/Extended/Finaldata"

print("Files in Finaldata folder:")
print(os.listdir(data_path))

Files in Finaldata folder:
['combined_conversations_conan_wildchat.csv']


In [ ]:


file_path = "/content/drive/MyDrive/GP2/NewTrustLens/Dataset/Extended/Finaldata/combined_conversations_conan_wildchat.csv"

df = pd.read_csv(file_path)

print("Shape of dataset:", df.shape)
print("\nColumns:")
print(df.columns)

df.head()

Shape of dataset: (12151, 10)

Columns:
Index(['dialogue_id', 'turn_id', 'text', 'MachineGen', 'assistant_type',
       'dataorigine', 'source', 'turnCountPerdialog', 'type', 'Explanation'],
      dtype='object')


,dialogue_id,turn_id,text,MachineGen,assistant_type,dataorigine,source,turnCountPerdialog,type,Explanation
0,0,0,We’ve just imported 20k ticking time bombs fro...,0,human,dialo_gold,human,8.0,assistant,accurate and balance responce
1,0,1,Surely people who also risked their lives to h...,0,human,dialo_gold,human,8.0,user,accurate and balance responce
2,0,2,Our own must always come first. We have at lea...,0,human,dialo_gold,human,8.0,assistant,accurate and balance responce
3,0,3,We also have at least 1 million empty homes. T...,0,human,dialo_gold,human,8.0,user,accurate and balance responce
4,0,4,Our soldiers are left to rot on our streets wh...,0,human,dialo_gold,human,8.0,assistant,accurate and balance responce


Here we will load the training/testing dataset.

### Dataset input (TrustLens)

Instead of providing separate train/test CSV files, we use **one combined CSV** and create the split inside the notebook.

#### What you may need to edit manually
In the configuration below, confirm the column names used in your CSV:
- `dialogue_id` (conversation identifier)
- `turn_id` (turn order inside the conversation)
- `text` (turn content)
- `label` (AI vs Human)

If your dataset uses different names (e.g., `conversation_id`, `turn`, `conversation`, `machine_generated`), update the config variables only.

In [ ]:
# -----------------------------
# Configuration (EDIT HERE ONLY)
# -----------------------------
SOURCE_CSV = "/content/drive/MyDrive/GP2/NewTrustLens/Dataset/Extended/Finaldata/combined_conversations_conan_wildchat.csv"

# Column names in your dataset
COL_DIALOGUE_ID = "dialogue_id"
COL_TURN_ID     = "turn_id"
COL_TEXT        = "text"
COL_LABEL       = "MachineGen"   # <-- IMPORTANT (your label column)
COL_PREV_TEXT   = "prev_text"    # generated in this notebook

USE_PREVIOUS_TURN = True

TEST_SIZE = 0.2
RANDOM_STATE = 42

# Output locations (Drive)
OUT_DIR = "/content/drive/MyDrive/GP2/NewTrustLens/ModelOutputs"
os.makedirs(OUT_DIR, exist_ok=True)

training_file = f"{OUT_DIR}/con_train_generated.csv"
Testing_file  = f"{OUT_DIR}/con_test_generated.csv"

# -----------------------------
# Load dataset
# -----------------------------
df = pd.read_csv(SOURCE_CSV, low_memory=False)

# Basic sanity checks
required_cols = [COL_DIALOGUE_ID, COL_TURN_ID, COL_TEXT, COL_LABEL]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(
        f"Missing required columns in CSV: {missing}\nAvailable columns: {list(df.columns)}"
    )

# Keep only needed columns
df = df[[COL_DIALOGUE_ID, COL_TURN_ID, COL_TEXT, COL_LABEL]].copy()

# Ensure types are valid (prevents vectorizer NaN error)
df[COL_TEXT] = df[COL_TEXT].fillna("").astype(str)
df[COL_TURN_ID] = pd.to_numeric(df[COL_TURN_ID], errors="coerce")
df[COL_LABEL] = pd.to_numeric(df[COL_LABEL], errors="coerce")

# Drop invalid rows
df = df.dropna(subset=[COL_DIALOGUE_ID, COL_TURN_ID, COL_LABEL]).copy()
df[COL_TURN_ID] = df[COL_TURN_ID].astype(int)
df[COL_LABEL] = df[COL_LABEL].astype(int)

# Sort to ensure correct previous-turn generation
df = df.sort_values([COL_DIALOGUE_ID, COL_TURN_ID]).reset_index(drop=True)

# Create previous-turn text (optional)
if USE_PREVIOUS_TURN:
    df[COL_PREV_TEXT] = df.groupby(COL_DIALOGUE_ID)[COL_TEXT].shift(1).fillna("").astype(str)
else:
    df[COL_PREV_TEXT] = ""

# -----------------------------
# Split by dialogue_id (prevents leakage)
# -----------------------------
unique_dialogues = df[COL_DIALOGUE_ID].drop_duplicates().values
train_ids, test_ids = train_test_split(
    unique_dialogues,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True
)

train_df = df[df[COL_DIALOGUE_ID].isin(train_ids)].copy()
test_df  = df[df[COL_DIALOGUE_ID].isin(test_ids)].copy()

print("Train shape:", train_df.shape, "| Unique dialogues:", train_df[COL_DIALOGUE_ID].nunique())
print("Test shape :", test_df.shape,  "| Unique dialogues:", test_df[COL_DIALOGUE_ID].nunique())

print("\nTrain columns:", list(train_df.columns))
print("Test columns :", list(test_df.columns))

# Save generated splits
train_df.to_csv(training_file, index=False)
test_df.to_csv(Testing_file, index=False)

print("\nGenerated files:")
print("training_file =", training_file)
print("Testing_file  =", Testing_file)

Train shape: (9794, 5) | Unique dialogues: 1264
Test shape : (2357, 5) | Unique dialogues: 317

Train columns: ['dialogue_id', 'turn_id', 'text', 'MachineGen', 'prev_text']
Test columns : ['dialogue_id', 'turn_id', 'text', 'MachineGen', 'prev_text']

Generated files:
training_file = /content/drive/MyDrive/GP2/NewTrustLens/ModelOutputs/con_train_generated.csv
Testing_file  = /content/drive/MyDrive/GP2/NewTrustLens/ModelOutputs/con_test_generated.csv


In [ ]:
class ItemSelector(BaseEstimator, TransformerMixin):
    """
    Selects a single column from a pandas DataFrame and always returns valid strings.
    This avoids TF-IDF / vectorizer crashes caused by NaN values.
    """
    def __init__(self, key):
        self.key = key

    def fit(self, x, y=None):
        return self

    def transform(self, data_dict):
        return data_dict[self.key].fillna("").astype(str)

Function start training

### Training function (TrustLens)

This function follows the original lab flow:
1) Load train/test CSV files  
2) Vectorize text features  
3) Train a baseline classifier (Logistic Regression)  
4) Evaluate using a classification report  
5) Save predictions for later analysis and UI integration

**Model choice:** Logistic Regression + TF-IDF (strong baseline for text classification).

In [ ]:
def start_training(training_file, Testing_file, baseline_classifier):
    """
    TrustLens baseline training (lab-style):
    1) Load train/test CSV files generated from a dialogue-level split
    2) Vectorize text features (TF-IDF)
    3) Train Logistic Regression baseline
    4) Evaluate using classification report
    5) Save model + predictions for documentation and later integration
    """

    # -----------------------------
    # 1) Load train/test
    # -----------------------------
    training_data = pd.read_csv(training_file, low_memory=False)
    test_data = pd.read_csv(Testing_file, low_memory=False)

    print("Training shape:", training_data.shape)
    print("Testing shape :", test_data.shape)
    print("Train columns :", list(training_data.columns))

    # -----------------------------
    # 2) Ensure text fields are valid strings
    # -----------------------------
    training_data[COL_TEXT] = training_data[COL_TEXT].fillna("").astype(str)
    test_data[COL_TEXT] = test_data[COL_TEXT].fillna("").astype(str)

    if USE_PREVIOUS_TURN:
        training_data[COL_PREV_TEXT] = training_data[COL_PREV_TEXT].fillna("").astype(str)
        test_data[COL_PREV_TEXT] = test_data[COL_PREV_TEXT].fillna("").astype(str)

    # -----------------------------
    # 3) Labels
    # -----------------------------
    Y_train = training_data[COL_LABEL].values
    Y_eval  = test_data[COL_LABEL].values

    # -----------------------------
    # 4) Feature pipelines (TF-IDF)
    # -----------------------------
    turn_tfidf = Pipeline([
        ("selector", ItemSelector(key=COL_TEXT)),
        ("tfidf", TfidfVectorizer(use_idf=True, ngram_range=(1, 2)))
    ])

    feats = [("turn_text", turn_tfidf)]

    if USE_PREVIOUS_TURN:
        prev_tfidf = Pipeline([
            ("selector", ItemSelector(key=COL_PREV_TEXT)),
            ("tfidf", TfidfVectorizer(use_idf=True, ngram_range=(1, 2)))
        ])
        feats.append(("prev_text", prev_tfidf))

    ppl = Pipeline([
        ("feats", FeatureUnion(feats)),
        ("clf", LogisticRegression(random_state=RANDOM_STATE, solver="lbfgs", max_iter=1000))
    ])

    # -----------------------------
    # 5) Train + Predict
    # -----------------------------
    model = ppl.fit(training_data, Y_train)
    y_pred = model.predict(test_data)

    print("\nClasses:", model.classes_)
    print("\nClassification Report:")
    print(classification_report(Y_eval, y_pred))

    # -----------------------------
    # 6) Save model + predictions
    # -----------------------------
    model_path = f"{OUT_DIR}/conversation_logistic_regression.joblib"

    joblib.dump(model, model_path)
    print("Saved model to:", model_path)

    prepare_file_output(y_pred, test_data, baseline_classifier)

    return model

### Save predictions output (TrustLens)

This helper saves predictions in a structured file that can be reused later for:
- error analysis
- reporting
- UI integration (showing predicted label per turn)

Output includes: dialogue_id, text, ground_truth_label, predicted_label.

In [ ]:
def prepare_file_output(y_pred, test_df, baseline_classifier):
    """
    Save predictions in a structured format for:
    - documentation (baseline results)
    - error analysis
    - later integration in TrustLens UI
    """


    out_path = f"{OUT_DIR}/predictions.csv"

    out_df = test_df[[COL_DIALOGUE_ID, COL_TURN_ID, COL_TEXT, COL_LABEL]].copy()

    # Keep previous turn in output if enabled
    if USE_PREVIOUS_TURN and COL_PREV_TEXT in test_df.columns:
        out_df[COL_PREV_TEXT] = test_df[COL_PREV_TEXT].fillna("").astype(str)

    out_df = out_df.rename(columns={COL_LABEL: "ground_truth"})
    out_df["predicted_label"] = y_pred

    out_df.to_csv(out_path, index=False)
    print("Saved predictions to:", out_path)
    print("Output shape:", out_df.shape)

### Run the training function (TrustLens)

1) Update `dataset_file` to your CSV path on Google Drive  
2) Choose a clear `baseline_classifier` name  
3) Run `start_training()` to train and evaluate the baseline model

In [ ]:
baseline_classifier = "baseline"
model = start_training(training_file, Testing_file, baseline_classifier)

Training shape: (9794, 5)
Testing shape : (2357, 5)
Train columns : ['dialogue_id', 'turn_id', 'text', 'MachineGen', 'prev_text']

Classes: [0 1]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.98      0.96      1279
           1       0.98      0.92      0.95      1078

    accuracy                           0.95      2357
   macro avg       0.96      0.95      0.95      2357
weighted avg       0.95      0.95      0.95      2357

Saved model to: /content/drive/MyDrive/GP2/NewTrustLens/ModelOutputs/conversation_logistic_regression.joblib
Saved predictions to: /content/drive/MyDrive/GP2/NewTrustLens/ModelOutputs/predictions.csv
Output shape: (2357, 6)


In [ ]:
import joblib

m = joblib.load("drive/MyDrive/GP2/NewTrustLens/ModelOutputs/conversation_logistic_regression.joblib")  # عدّلي المسار لملفك
print("TYPE:", type(m))

if hasattr(m, "named_steps"):
    print("✅ PIPELINE detected")
    print("STEPS:", m.named_steps.keys())
else:
    print("❌ Not a Pipeline (likely classifier only)")

TYPE: <class 'sklearn.pipeline.Pipeline'>
✅ PIPELINE detected
STEPS: dict_keys(['feats', 'clf'])
